# 1. Import Libraries

In [61]:
!pip install ultralytics wandb opencv-python pyyaml

In [62]:
import os
import numpy as np
import pandas as pd
import yaml
import zipfile
import matplotlib.pyplot as plt
import cv2
from IPython.display import HTML
from matplotlib import animation
from tqdm import tqdm
from PIL import Image
from ultralytics import YOLO
from IPython.display import FileLink

# 3. YAML File 

In [63]:
data_yaml = dict(
    train= '/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/train/images',
    val= '/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/val/images',
    test= '/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images',
    nc = 9,
    names =['Trafic Light Signal', 'Stop Signal', 'Speedlimit Signal', 'Crosswalk Signal', 'Crosswalk', 'Pedestrian', 'Bus', 'Car', 'Truck']
)

In [64]:
with open('dataset.yaml', 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)

# 4. Dataset Import

In [65]:
def create_zip(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zip_ref:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                zip_ref.write(file_path, arcname=os.path.relpath(file_path, source_folder))

In [66]:
def extract_zip(zip_file, destination_folder):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(destination_folder)

# 5. Train

In [67]:
model = YOLO("yolo11m.pt") 

In [68]:
#!yolo task=detect mode=train model=yolo11m.pt data=dataset.yaml epochs=25 imgsz=640
!yolo task=detect mode=train model=yolo11m.pt data=dataset.yaml epochs=1 imgsz=50

Ultralytics 8.3.88 🚀 Python-3.12.9 torch-2.6.0 CPU (Apple M4 Pro)
engine/trainer: task=detect, mode=train, model=yolo11m.pt, data=dataset.yaml, epochs=1, time=None, patience=100, batch=16, imgsz=50, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=None, format=

In [69]:
!zip -r runs.zip runs/

updating: runs/ (stored 0%)
updating: runs/detect/ (stored 0%)
updating: runs/detect/train/ (stored 0%)
updating: runs/detect/train/args.yaml (deflated 53%)
updating: runs/detect/train/weights/ (stored 0%)
updating: runs/detect/train2/ (stored 0%)
updating: runs/detect/train2/args.yaml (deflated 53%)
updating: runs/detect/train2/weights/ (stored 0%)
updating: runs/.DS_Store (deflated 97%)
updating: runs/detect/train3/ (stored 0%)
updating: runs/detect/train3/args.yaml (deflated 53%)
updating: runs/detect/train3/weights/ (stored 0%)
updating: runs/detect/.DS_Store (deflated 96%)
updating: runs/detect/train4/ (stored 0%)
updating: runs/detect/train4/.DS_Store (deflated 96%)
updating: runs/detect/train4/args.yaml (deflated 53%)
updating: runs/detect/train4/weights/ (stored 0%)
updating: runs/detect/train4/labels_correlogram.jpg (deflated 33%)
updating: runs/detect/train4/train_batch0.jpg (deflated 4%)
updating: runs/detect/train4/train_batch1.jpg (deflated 4%)
updating: runs/detect/train4

In [70]:
FileLink('runs.zip')

/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/runs.zip

# 6. Results

In [71]:
def display_training_results(directory, file_extension=('.jpg', '.png'), images_per_row=2, image_height=10):
    image_paths = []
    for dirname, _, filenames in os.walk(directory):
        for filename in filenames:
            if filename.endswith(file_extension):
                image_paths.append(os.path.join(dirname, filename))
    image_paths = sorted(image_paths)

    num_images = len(image_paths)
    num_rows = (num_images + images_per_row - 1) // images_per_row  
    figsize = (images_per_row * image_height, num_rows * image_height)
    
    fig, axes = plt.subplots(num_rows, images_per_row, figsize=figsize)
    axes = axes.flatten() 

    for i, path in enumerate(image_paths):
        image = Image.open(path)
        axes[i].imshow(np.array(image))
        axes[i].axis('off') 

    for j in range(len(image_paths), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [84]:
display_training_results('/runs/detect/train')

ValueError: Number of rows must be a positive integer, not 0

<Figure size 2000x0 with 0 Axes>

# 7. Predict

In [87]:
best_path = 'runs/detect/train5/weights/best.pt'
source = 'dataset/test/images'

image_paths = []
for dirname, _, filenames in os.walk(source):
    for filename in filenames:
        if filename.endswith('.jpg') or filename.endswith('.png'): 
            image_paths.append(os.path.join(dirname, filename))
image_paths = sorted(image_paths)

model = YOLO(best_path)

In [88]:
!yolo task=detect mode=predict model={best_path} conf=0.1 source={source}

Ultralytics 8.3.88 🚀 Python-3.12.9 torch-2.6.0 CPU (Apple M4 Pro)
YOLO11m summary (fused): 125 layers, 20,036,971 parameters, 0 gradients, 67.7 GFLOPs

image 1/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1040.jpg: 64x64 7 Crosswalks, 7.0ms
image 2/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1042.jpg: 64x64 6 Crosswalks, 5.9ms
image 3/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1043.jpg: 64x64 6 Crosswalks, 6.0ms
image 4/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1054.jpg: 64x64 1 Trafic Light Signal, 4 Crosswalks, 4 Pedestrians, 6.1ms
image 5/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1056.jpg: 64x64 3 Speedlimit Signals, 6.0ms
image 6/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1057.jpg: 64x64 1 Crosswalk, 4 Pedestrians, 2 Cars, 2 Trucks, 6.3ms
image 7/167 /U

In [89]:
results = model.predict(source, conf=0.1)
results


image 1/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1040.jpg: 64x64 7 Crosswalks, 8.1ms
image 2/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1042.jpg: 64x64 6 Crosswalks, 11.2ms
image 3/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1043.jpg: 64x64 6 Crosswalks, 7.8ms
image 4/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1054.jpg: 64x64 1 Trafic Light Signal, 4 Crosswalks, 4 Pedestrians, 8.9ms
image 5/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1056.jpg: 64x64 3 Speedlimit Signals, 12.0ms
image 6/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1057.jpg: 64x64 1 Crosswalk, 4 Pedestrians, 2 Cars, 2 Trucks, 11.5ms
image 7/167 /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images/i1068.jpg: 64x64 8 Crosswalks, 8.3ms
image 8/167 /Users/jorgecunha/Pychar

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Trafic Light Signal', 1: 'Stop Signal', 2: 'Speedlimit Signal', 3: 'Crosswalk Signal', 4: 'Crosswalk', 5: 'Pedestrian', 6: 'Bus', 7: 'Car', 8: 'Truck'}
 obb: None
 orig_img: array([[[215, 215, 179],
         [213, 213, 177],
         [210, 210, 174],
         ...,
         [190, 198, 167],
         [189, 197, 166],
         [189, 197, 166]],
 
        [[224, 224, 188],
         [223, 223, 187],
         [221, 221, 185],
         ...,
         [189, 197, 166],
         [188, 196, 165],
         [188, 196, 165]],
 
        [[227, 227, 191],
         [227, 227, 191],
         [228, 228, 192],
         ...,
         [188, 196, 165],
         [187, 195, 164],
         [187, 195, 164]],
 
        ...,
 
        [[190, 191, 149],
         [185, 186, 144],
         [182, 182, 142],
         ...,
         [158, 146, 112],
         [156, 144, 11

In [90]:
paths = []

for dirname, _, filenames in os.walk(source):
    for filename in filenames:
        if filename[-4:] == '.jpg' or filename[-4:] == '.png':
            paths += [(os.path.join(dirname, filename))]
            
paths = sorted(paths)

In [91]:
df = pd.DataFrame(columns = range(6))

for i in range(len(results)):
    arri = pd.DataFrame(results[i].boxes.data.cpu().numpy()).astype(float)
    path = paths[i]
    file = path.split('/')[-1]
    arri = arri.assign(file = file)
    arri = arri.assign(i = i)
    df = pd.concat([df, arri], axis = 0)
    
df.columns=['x','y','x2','y2','confidence','class','file','i']
display(df)

/var/folders/lf/hbzd31cs7kbgcfwrry3vrgdw0000gn/T/ipykernel_3205/3877240528.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, arri], axis = 0)


,x,y,x2,y2,confidence,class,file,i
0,138.105896,6.429481,415.032257,416.000000,0.995104,4.0,i1040.jpg,0.0
1,24.104803,3.751514,405.342926,411.028229,0.986213,4.0,i1040.jpg,0.0
2,0.000000,1.450514,300.205292,388.966736,0.930998,4.0,i1040.jpg,0.0
3,216.241089,12.399601,416.000000,416.000000,0.925302,4.0,i1040.jpg,0.0
4,0.000000,0.713863,187.330460,416.000000,0.775310,4.0,i1040.jpg,0.0
...,...,...,...,...,...,...,...,...
11,126.494179,17.928846,161.592743,92.338623,0.114270,0.0,i996.jpg,165.0
0,143.425537,8.611873,416.000000,416.000000,0.999012,4.0,i997.jpg,166.0
1,35.109604,0.000000,406.532471,411.967163,0.997295,4.0,i997.jpg,166.0
2,240.791977,10.576931,416.000000,416.000000,0.976276,4.0,i997.jpg,166.0


In [92]:
def draw_box(image_index):
    image_path = paths[image_index]
    image = cv2.imread(image_path)
    height, width = image.shape[:2]
    filename = image_path.split('/')[-1]
    
    if not df[df['file'] == filename].empty:
        boxes = df[df['file'] == filename].reset_index(drop=True)

        for box_index in range(len(boxes)):
            label = boxes.loc[box_index, 'class']
            x = int(boxes.loc[box_index, 'x'])
            y = int(boxes.loc[box_index, 'y'])
            x2 = int(boxes.loc[box_index, 'x2'])
            y2 = int(boxes.loc[box_index, 'y2'])
            
            cv2.putText(
                image, f'{label}', (x, int(y - 4)),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2
            )
            cv2.rectangle(image, (x, y), (x2, y2), (0, 255, 0), 2) 
    
    return image

In [93]:
def create_animation(images):
    fig = plt.figure(figsize=(12, 8))
    image_display = plt.imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))
    text = plt.text(0.05, 0.05, f'Slide {0}', transform=fig.transFigure, fontsize=14, color='blue')
    plt.axis('off')
    plt.close()

    def animate_func(frame_index):
        image_display.set_array(cv2.cvtColor(images[frame_index], cv2.COLOR_BGR2RGB))
        text.set_text(f'Slide {frame_index}')
        return [image_display]

    return animation.FuncAnimation(fig, animate_func, frames=len(images), interval=1000)

In [94]:
annotated_images = []

for index in tqdm(range(len(paths))):
    annotated_images.append(draw_box(index))

100%|██████████| 167/167 [00:00<00:00, 473.75it/s]
